# EasyPush: conedir-success vs L2C-failure (minimal Colab confirmation)

Binary-NCE contrastive RL, **no causal changes**. Two 100k runs, seed 1:

1. **conedir** (`fetch_push_easy_conedir`) - fixed -x gripper, goal in a +/-60 deg
   cone around +x. State-based calibration rung inspired by the original image-Push
   geometry (NOT an exact reproduction). *Expected: high, uniform across cone bins.*
2. **L2C** (`fetch_push_easy_neutral_dir`) - fixed -x gripper, goal in ANY direction,
   so the policy must select the contact side from the goal. *Expected: fails; a
   fixed +x / contact-side shortcut; -x quadrant near 0.*

Each run reports: success curve, random floor + scripted-oracle ceiling, success by
cone-bin (conedir) / quadrant (L2C), min/final object-goal distance, a
displacement-direction probe, and a montage GIF. Artifacts save to Drive.

Run cells top-to-bottom: conedir trains+reports first, then L2C.

In [ ]:
# 1. Install dependencies (~2 min) + headless MuJoCo rendering backend.
!pip -q install "jax[cuda12]" dm-haiku optax gymnasium gymnasium-robotics mujoco imageio
import os
os.environ["MUJOCO_GL"] = "egl"
import jax; print("JAX devices:", jax.devices())

In [ ]:
# 2. Get / update the code (your fork).
import os
os.chdir("/content")
if not os.path.exists("/content/contrastive_rl"):
    !git clone https://github.com/tingrui-huang/contrastive_rl.git
%cd /content/contrastive_rl
!git pull

In [ ]:
# 3. Mount Google Drive (checkpoints + reports are written here).
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 4. Params + guard-railed binary-NCE config builder + report helpers.
import os, json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
from crl.config import Config
from crl.train import train
from crl import report_push, visualize

SEED            = 1
STEPS           = 100_000
REPORT_EPISODES = 100
BASE            = "/content/drive/MyDrive/easypush_colab"   # all artifacts here
os.makedirs(BASE, exist_ok=True)
print("BASE:", BASE, "| SEED:", SEED, "| STEPS:", STEPS)

def build_nce_config(env_name, seed, ckpt_dir):
    """Binary-NCE, no causal changes, fresh run. Matches the seed-0 reference."""
    cfg = Config(
        env_name=env_name,
        use_td=False, twin_q=False,            # binary NCE (contrastive_nce)
        random_goals=0.5, entropy_coefficient=0.0,
        max_number_of_steps=STEPS,
        min_replay_size=5_000, random_steps=5_000,
        num_sgd_steps_per_step=4, batch_size=256,
        eval_every_steps=10_000, eval_episodes=50, log_every_steps=20_000,
        seed=seed, ckpt_dir=ckpt_dir, resume=False,
    )
    assert cfg.use_td is False and cfg.twin_q is False, "must be binary NCE (not TD/C-learning)"
    assert cfg.random_goals == 0.5 and cfg.entropy_coefficient == 0.0, "config drift"
    assert cfg.resume is False, "formal confirmation runs start fresh"
    return cfg

def plot_curves(run_dir, tag):
    m = json.load(open(os.path.join(run_dir, "metrics.json")))
    steps = [r["step"] for r in m]
    fig, ax = plt.subplots(1, 2, figsize=(9, 3))
    ax[0].plot(steps, [r["success"] for r in m], "-o")
    ax[0].set_title(tag + " success"); ax[0].set_ylim(-0.02, 1.02)
    ax[0].set_xlabel("env steps"); ax[0].grid(alpha=0.3)
    ax[1].plot(steps, [r["final_dist"] for r in m], "-o", color="tab:red", label="final")
    ax[1].plot(steps, [r["min_dist"] for r in m], "--o", color="tab:orange", label="min")
    ax[1].set_title("object-goal distance"); ax[1].set_xlabel("env steps")
    ax[1].legend(); ax[1].grid(alpha=0.3)
    fig.tight_layout(); plt.show()

def make_report(env_name, run_dir, tag):
    ckpt = os.path.join(run_dir, "best.pkl")
    rep = report_push.evaluate_report(env_name, ckpt, episodes=REPORT_EPISODES, seed=777)
    report_push.print_report(rep)
    out = os.path.join(BASE, f"report_{tag}.json")
    json.dump(rep, open(out, "w"), indent=2); print("saved", out)
    plot_curves(run_dir, tag)
    gif = os.path.join(BASE, f"montage_{tag}.gif")
    visualize.rollout_gif(env_name, ckpt=ckpt, out=gif, episodes=6, seed=1)
    display(Image(gif))
    return rep

In [ ]:
# 5. Train conedir (calibration rung), seed 1, 100k. ~30-50 min on T4.
ENV_CONEDIR = "fetch_push_easy_conedir"
RUN_CONEDIR = os.path.join(BASE, f"{ENV_CONEDIR}_s{SEED}")
cfg = build_nce_config(ENV_CONEDIR, SEED, RUN_CONEDIR)
print(f"OK: {ENV_CONEDIR}, binary NCE, seed {SEED} -> {RUN_CONEDIR}")
train(cfg)

In [ ]:
# 6. Report conedir: numeric report + curves + montage.
rep_conedir = make_report(ENV_CONEDIR, RUN_CONEDIR, "conedir_s1")

In [ ]:
# 7. Train L2C (neutral-dir), seed 1, 100k. Run AFTER conedir finishes.
ENV_L2C = "fetch_push_easy_neutral_dir"
RUN_L2C = os.path.join(BASE, f"{ENV_L2C}_s{SEED}")
cfg = build_nce_config(ENV_L2C, SEED, RUN_L2C)
print(f"OK: {ENV_L2C}, binary NCE, seed {SEED} -> {RUN_L2C}")
train(cfg)

In [ ]:
# 8. Report L2C: numeric report + curves + montage.
rep_l2c = make_report(ENV_L2C, RUN_L2C, "l2c_s1")

## Contrast summary
The structural claim: **conedir succeeds uniformly across cone bins**, while
**L2C fails with a fixed +x / contact-side shortcut** (`cos(+x) >= cos(goal)`,
`shortcut=True`) and near-zero success in the -x quadrant.

In [ ]:
# 9. Side-by-side conedir vs L2C.
c = json.load(open(os.path.join(BASE, "report_conedir_s1.json")))
l = json.load(open(os.path.join(BASE, "report_l2c_s1.json")))
print("=" * 68)
print("CONTRAST  conedir (calibration)  vs  L2C (neutral-dir)   seed 1, 100k")
print("=" * 68)
for tag, r in [("conedir", c), ("L2C", l)]:
    d = r["displacement_probe"]
    print(f"{tag:8s} trained={r['trained']['success']:.3f}  "
          f"random={r['random']['success']:.3f}  oracle={r['oracle']['success']:.3f}  "
          f"cos_goal={d['cos_disp_goal_mean']:.2f}  cos_+x={d['cos_disp_posx_mean']:.2f}  "
          f"shortcut={d['shortcut_flag']}")
print()
print("conedir success by cone bin:")
for k, v in c["success_by_cone_bin"].items():
    print(f"   {k:8s}: success={v['success']}  (n={v['n']})")
print("L2C success by quadrant:")
for k, v in l["success_by_quadrant"].items():
    print(f"   {k:28s}: success={v['success']}  (n={v['n']})")
print("L2C +x vs -x asymmetry:", l["plus_minus_x_asymmetry"])